# Headers

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import sys
import os

import comet_ml
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from xgboost import XGBClassifier

try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()
    
parent_dir = os.path.join(current_dir, '..', '..', '..')
sys.path.insert(0, parent_dir)
    
    
from analysis_scripts.config import CFG


CFG.seed_all(CFG.seed)
sns.set_theme(style='ticks')

# Load preprocessed data

In [ ]:
train_df = pd.read_csv('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/Data/proc_df_1.csv').drop('Unnamed: 0', axis=1)
test_df = pd.read_csv('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/Data/proc_df_2.csv').drop('Unnamed: 0', axis=1)

assert not train_df.isna().sum().sum(), 'Train, data containes NaNs!!'
assert not test_df.isna().sum().sum(), 'Test, data containes NaNs!!'

# Prepare Data

Exclude variables that affect background shape: 
* is_mass_window,
* mass_Lc,
* mass_pt_correlation
* ?Mt_Lc?

In [ ]:
# Comet logger
# experiment = comet_ml.Experiment(
#     project_name="SPD ML",
#     workspace="artem-smirnov",
#     api_key="yyKk8vacVuy8dbttSCIgF2iQO"
# )

features_list = CFG.feature_set_3

try:
    experiment.set_name('feature_set_1, params_xgboost_1')
    formatted_text = json.dumps({"feature_names": features_list}, indent=2)
    experiment.log_text(text=formatted_text)
except NameError:
    pass

train_df = train_df[features_list + [CFG.target_name, 'mass_Lc']].copy()
test_df = test_df[features_list + [CFG.target_name, 'mass_Lc']].copy()

x = train_df.drop(['tag', 'mass_Lc'], axis=1).copy()
y = train_df['tag'].map({'Sig': 1, 'Bg': 0}).copy()

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, shuffle=True, stratify=y, random_state=CFG.seed)

x_test = test_df.drop(['tag', 'mass_Lc'], axis=1).copy()
y_test = test_df['tag'].map({'Sig': 1, 'Bg': 0}).copy()

In [ ]:
# Feature encoding
if features_list == CFG.feature_set_1:
    cols_to_encode = [
        'highest_pt_daughter', 'highest_momentum_daughter', 'highest_OA_daughter',
        'highest_eta_daughter', 'highest_dca_Lc_daughter', 'highest_dca_PV_daughter',
    ]
elif features_list == CFG.feature_set_2:
    cols_to_encode = [
        'highest_momentum_daughter',
        'highest_eta_daughter', 'highest_dca_Lc_daughter', 'highest_dca_PV_daughter',
    ]
elif features_list == CFG.feature_set_3:
    cols_to_encode = []

preprocessor = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(sparse_output=False, dtype=int), cols_to_encode)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor.fit(x_train)
x_train_encoded = preprocessor.transform(x_train)
x_val_encoded = preprocessor.transform(x_val)
x_test_encoded = preprocessor.transform(x_test)

feature_names = preprocessor.get_feature_names_out()

x_train = pd.DataFrame(x_train_encoded, columns=feature_names, index=x_train.index)
x_val = pd.DataFrame(x_val_encoded, columns=feature_names, index=x_val.index)
x_test = pd.DataFrame(x_test_encoded, columns=feature_names, index=x_test.index)


# Drop excluded columns
if features_list == CFG.feature_set_2:
    
    features_to_drop = [
        'highest_momentum_daughter_P_pip', 'highest_momentum_daughter_P_p',
        'highest_eta_daughter_eta_pip', 'highest_dca_PV_daughter_dist_pip_PV_xy',
        'highest_dca_Lc_daughter_dist_p_Lc_xy',
    ]

    x_train = x_train.drop(columns=features_to_drop)
    x_val = x_val.drop(columns=features_to_drop)
    x_test = x_test.drop(columns=features_to_drop)

# Stacking

In [ ]:
from analysis_scripts.custom_cv import CrossVal

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score


######################################################################################
# First Layer

models_list = [
    XGBClassifier,
    CatBoostClassifier,
    LGBMClassifier,
    LogisticRegression,
]

# feature set 9
params_list = [
    {},
    {},
    {},
    {},
]
params_update_list = [
    #xgboost
    {
        'booster': 'gbtree',
        'grow_policy': 'depthwise',
        'tree_method': 'hist',
        'random_state': CFG.seed,
        'eval_metric': 'auc',
        'verbosity': 0,
        'device': 'cuda' if CFG.gpu_available else 'cpu',
        'n_jobs': 1 if CFG.gpu_available else 6,
        'max_bin': 256, 
    },
    # catboost
    {
        'loss_function': 'Logloss',
        # 'grow_policy': 'SymmetricTree',
        'eval_metric': 'AUC',
        'use_best_model': True,
        'task_type': 'GPU' if CFG.gpu_available else 'CPU',
        'border_count': 256,
        'verbose': 0,
        'thread_count': 1 if CFG.gpu_available else 6,
        'random_seed': CFG.seed,
    },
    # lightgbm
    {
        'boosting_type': 'gbdt',
        'random_state': CFG.seed,
        'n_jobs': 1 if CFG.gpu_available else 6,   
        'verbose': -1,
        'device_type': 'cuda' if CFG.gpu_available else 'cpu'
    },
    # LogisticRegression
    {
        'penalty': 'l2',
        'solver': 'newton-cholesky',
        'fit_intercept': True,
        'verbose': 0,
        'n_jobs': 6,
        'random_state': CFG.seed
    },
]

for idx in range(len(params_list)):
    params_list[idx].update(params_update_list[idx])

oof_pred_proba_list = []            # (n_models, len(y_train)), from train sample, input for second layer training
first_layer_pred_proba_val = []     # (n_models, len(y_val)), from val sample, input for second layer validation
first_layer_pred_proba_test = []    # (n_models, len(y_test)), from val sample, input for second layer test
first_layer_models = []

for model, params in zip(models_list, params_list):
    
    print(f'\nProcess {model.__name__}: \n')
    
    ###############################################
    # Prepare second layer input data
    if model.__name__ == 'LogisticRegression':
        
        current_model = make_pipeline(
            StandardScaler(),
            LogisticRegression(**params)
        )
    
    else:
        current_model = model(**params)
    
    cv = CrossVal(
        x_train=x_train,
        y_train=y_train,
        nfolds=5,
        stratification=True
    )

    _, oof_pred_proba = cv.fit(
        model=current_model,
        eval_metric=roc_auc_score,
        train_validation=True
    )
    
    oof_pred_proba_list.append(oof_pred_proba)
    
    ###############################################
    # Train first layer model
    if current_model.__class__.__name__ == 'XGBClassifier':
    
        current_model.fit(
            x_train, 
            y_train,
            eval_set=[(x_train, y_train), (x_val, y_val)],
            verbose=0
        )
        
        first_layer_pred_proba_val.append(current_model.predict_proba(x_val)[:, 1])
        first_layer_pred_proba_test.append(current_model.predict_proba(x_test)[:, 1])
    
    elif current_model.__class__.__name__ == 'LGBMClassifier':

        current_model.fit(
            x_train,
            y_train,
            eval_metric='auc',
            eval_set=[(x_val, y_val)],
        )
    
        first_layer_pred_proba_val.append(current_model.predict_proba(x_val)[:, 1])
        first_layer_pred_proba_test.append(current_model.predict_proba(x_test)[:, 1])
        
    elif current_model.__class__.__name__ == 'CatBoostClassifier':

        current_model.fit(
            x_train,
            y_train,
            eval_set=(x_val, y_val),
            plot=False
        )
    
        first_layer_pred_proba_val.append(current_model.predict_proba(x_val)[:, 1])
        first_layer_pred_proba_test.append(current_model.predict_proba(x_test)[:, 1])
    
    elif current_model.__class__.__name__ in ['LogisticRegression', 'Pipeline']:
    
        current_model.fit(x_train, y_train)
    
        first_layer_pred_proba_val.append(current_model.predict_proba(x_val)[:, 1])
        first_layer_pred_proba_test.append(current_model.predict_proba(x_test)[:, 1])
    
    elif current_model.__class__.__name__ == 'GradientBoostedTreesLearner':

        train_ydf_df = x_train.copy()
        train_ydf_df['target'] = y_train.copy()
    
        current_model = current_model.train(train_ydf_df)
        
        first_layer_pred_proba_val.append(current_model.predict(x_val))
        first_layer_pred_proba_test.append(current_model.predict(x_test))
            
    elif current_model.__class__.__name__ == 'RandomForestLearner':
        
        train_ydf_df = x_train.copy()
        train_ydf_df['target'] = y_train.copy()
    
        current_model = current_model.train(train_ydf_df)
    
        first_layer_pred_proba_val.append(current_model.predict(x_val))
        first_layer_pred_proba_test.append(current_model.predict(x_test))
    
    first_layer_models.append(current_model)
        
        
oof_pred_proba_list = np.array(oof_pred_proba_list)
first_layer_pred_proba_val = np.array(first_layer_pred_proba_val)
first_layer_pred_proba_test = np.array(first_layer_pred_proba_test)


In [ ]:
np.save(file='/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/oof_pred_proba_list_set_2.npy', arr=oof_pred_proba_list)
np.save(file='/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/first_layer_pred_proba_val_set_2.npy', arr=first_layer_pred_proba_val)
np.save(file='/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/first_layer_pred_proba_test_set_2.npy', arr=first_layer_pred_proba_test)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, f1_score


######################################################################################
# Second Layer

oof_pred_proba_list = np.load('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/oof_pred_proba_list_set_2.npy')
first_layer_pred_proba_val = np.load('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/first_layer_pred_proba_val_set_2.npy')
first_layer_pred_proba_test = np.load('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/first_layer_pred_proba_test_set_2.npy')

x_train_2 = oof_pred_proba_list.T.copy()
x_val_2 = first_layer_pred_proba_val.T.copy()
x_test_2 = first_layer_pred_proba_test.T.copy()

# oof_pred_proba_list = np.load('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/oof_pred_proba_list_set_9.npy')
# first_layer_pred_proba_val = np.load('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/first_layer_pred_proba_val_set_9.npy')
# first_layer_pred_proba_test = np.load('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/analysis_notebooks/ML_analysis/first_layer_pred_proba_test_set_9.npy')
# 
# x_train_2 = np.concatenate([x_train_2, oof_pred_proba_list.T], axis=1)
# x_val_2 = np.concatenate([x_val_2, first_layer_pred_proba_val.T], axis=1)
# x_test_2 = np.concatenate([x_test_2, first_layer_pred_proba_test.T], axis=1)

x_train_2 = pd.DataFrame(x_train_2)
x_val_2 = pd.DataFrame(x_val_2)
x_test_2 = pd.DataFrame(x_test_2)

# scaler = StandardScaler()
# scaler.fit(x_train_2)
# x_train_2 = scaler.transform(x_train_2)
# x_val_2 = scaler.transform(x_val_2)
# x_test_2 = scaler.transform(x_test_2)

model = LogisticRegression(
    penalty='l2',
    solver='newton-cholesky',
    fit_intercept=True,
    verbose=0,
    n_jobs=6,
    random_state=CFG.seed
)

params_space = {
    'C': [1e-2, 1e-1, 1, 10],
    'tol': [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'max_iter': [100, 200, 300, 400],
}

searcher = GridSearchCV(
    estimator=model,
    param_grid=params_space,
    scoring='roc_auc',
    cv=5,
    n_jobs=6,
)

searcher.fit(x_train_2, y_train)

model = searcher.best_estimator_

y_pred_proba_train = model.predict_proba(x_train_2)[:, 1]
# y_pred_proba_train = x_train_2.mean(axis=1)
y_pred_train = np.where(y_pred_proba_train > 0.5, 1, 0)



print(f'\nTrain Sample')
print(f"ROC-AUC: {roc_auc_score(y_train, y_pred_proba_train):.4f}")
print(f"Precision: {precision_score(y_train, y_pred_train):.4f}")
print(f"F1-score: {f1_score(y_train, y_pred_train):.4f}")

y_pred_proba_val = model.predict_proba(x_val_2)[:, 1]
# y_pred_proba_val = x_val_2.mean(axis=1)
y_pred_val = np.where(y_pred_proba_val > 0.5, 1, 0)

print(f'\nVal Sample')
print(f"ROC-AUC: {roc_auc_score(y_val, y_pred_proba_val):.4f}")
print(f"Precision: {precision_score(y_val, y_pred_val):.4f}")
print(f"F1-score: {f1_score(y_val, y_pred_val):.4f}")

y_pred_proba_test = model.predict_proba(x_test_2)[:, 1]
# y_pred_proba_test = x_test_2.mean(axis=1)
y_pred_test = np.where(y_pred_proba_test > 0.5, 1, 0)

print(f'\nTest Sample')
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_test):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_test):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_test):.4f}")

#### Stacking check

In [ ]:
model.coef_

In [ ]:
from analysis_scripts.feature import FeatureEngineer


check_df = x_train_2.copy()
check_df['target'] = y_train.reset_index(drop=True)

feature_engin = FeatureEngineer()

feature_engin.features_correlation(
    df=check_df,
    columns=list(np.arange(6, 12, 1))
)

feature_engin.feature_importance_corr(
    df=check_df,
    target_name='target',
    continual_columns=list(np.arange(6, 12, 1)),
)

feature_engin.feature_importance_stat_test(
    df=check_df,
    target_name='target',
    features_to_f_classif_test=list(np.arange(6, 12, 1)),
    visualization=True
)

#### Classifier Calibration

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay


n_bins=20

fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(14, 16))
fig.suptitle('Classifier Calibration', fontweight='bold')

#######################################
sns.histplot(
    data=y_pred_proba_train,
    stat='density',
    bins=n_bins,
    ax=axes[0][0]
)

axes[0][0].set_title('Train sample', fontweight='bold')
axes[0][0].set_xlabel('Probabilities')
axes[0][0].set_ylabel('Density')
axes[0][0].grid(True, linestyle='--', alpha=0.8)
#######################################
disp = CalibrationDisplay.from_predictions(y_train, y_pred_proba_train, name='train', n_bins=n_bins, ax=axes[1][0])

axes[1][0].set_title('Calbration Plot', fontweight='bold')
# axes[1][0].set_xlabel('Probabilities', fontweight='bold')
# axes[1][0].set_ylabel('Density', fontweight='bold')
axes[1][0].grid(True, linestyle='--', alpha=0.8)
#######################################
display = RocCurveDisplay.from_predictions(
    y_true=y_train,
    y_pred=y_pred_proba_train,
    name=f"Train sample",
    curve_kwargs=dict(color="darkorange"),
    plot_chance_level=True,
    despine=True,
    ax=axes[2][0]
)

axes[2][0].set_title('ROC Curve', fontweight='bold')
axes[2][0].grid(True, linestyle='--', alpha=0.8)
#######################################
ConfusionMatrixDisplay.from_predictions(
    y_true=y_train, 
    y_pred=y_pred_train,
    display_labels=['sig', 'bkg'],
    cmap=plt.cm.Blues,
    values_format='d',
    ax=axes[3][0]
)

axes[3][0].set_title('Confusion Matrix (threshold=0.5)', fontweight='bold')
#######################################
sns.histplot(
    data=y_pred_proba_val,
    stat='density',
    bins=n_bins,
    ax=axes[0][1]
)

axes[0][1].set_title('Val sample', fontweight='bold')
axes[0][1].set_xlabel('Probabilities')
axes[0][1].set_ylabel('Density')
axes[0][1].grid(True, linestyle='--', alpha=0.8)
#######################################
disp = CalibrationDisplay.from_predictions(y_val, y_pred_proba_val, name='val', n_bins=n_bins, ax=axes[1][1])

axes[1][1].set_title('Calbration Plot', fontweight='bold')
# axes[1][0].set_xlabel('Probabilities', fontweight='bold')
# axes[1][0].set_ylabel('Density', fontweight='bold')
axes[1][1].grid(True, linestyle='--', alpha=0.8)
#######################################
display = RocCurveDisplay.from_predictions(
    y_true=y_val,
    y_pred=y_pred_proba_val,
    name=f"Val sample",
    curve_kwargs=dict(color="darkorange"),
    plot_chance_level=True,
    despine=True,
    ax=axes[2][1]
)

axes[2][1].set_title('ROC Curve', fontweight='bold')
axes[2][1].grid(True, linestyle='--', alpha=0.8)
#######################################
ConfusionMatrixDisplay.from_predictions(
    y_true=y_val, 
    y_pred=y_pred_val,
    display_labels=['sig', 'bkg'],
    cmap=plt.cm.Blues,
    values_format='d',
    ax=axes[3][1]
)

axes[3][1].set_title('Confusion Matrix (threshold=0.5)', fontweight='bold')
#######################################
sns.histplot(
    data=y_pred_proba_val,
    stat='density',
    bins=n_bins,
    ax=axes[0][2]
)

axes[0][2].set_title('Test sample', fontweight='bold')
axes[0][2].set_xlabel('Probabilities')
axes[0][2].set_ylabel('Density')
axes[0][2].grid(True, linestyle='--', alpha=0.8)
#######################################
disp = CalibrationDisplay.from_predictions(y_test, y_pred_proba_test, name='test', n_bins=n_bins, ax=axes[1][2])

axes[1][2].set_title('Calbration Plot', fontweight='bold')
# axes[1][0].set_xlabel('Probabilities', fontweight='bold')
# axes[1][0].set_ylabel('Density', fontweight='bold')
axes[1][2].grid(True, linestyle='--', alpha=0.8)
#######################################
display = RocCurveDisplay.from_predictions(
    y_true=y_test,
    y_pred=y_pred_proba_test,
    name=f"Test sample",
    curve_kwargs=dict(color="darkorange"),
    plot_chance_level=True,
    despine=True,
    ax=axes[2][2]
)

axes[2][2].set_title('ROC Curve', fontweight='bold')
axes[2][2].grid(True, linestyle='--', alpha=0.8)
#######################################
ConfusionMatrixDisplay.from_predictions(
    y_true=y_test, 
    y_pred=y_pred_test,
    display_labels=['sig', 'bkg'],
    cmap=plt.cm.Blues,
    values_format='d',
    ax=axes[3][2]
)

axes[3][2].set_title('Confusion Matrix (threshold=0.5)', fontweight='bold')
#######################################
plt.tight_layout()

try:
    experiment.log_figure(
        figure_name=f"Calibration plots",
        figure=plt.gcf()
    )
except NameError:
    pass

plt.show()

#### Optimize threshold

In [ ]:
from analysis_scripts.estimate_scripts import signal_estimates
from sklearn.metrics import f1_score, precision_score


pred_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred_proba': y_pred_proba_test
})

thresholds = np.arange(0.1, 1, 0.05)
f1_scores = []
precision_scores = []
s_over_b = []
overall_s_sqrt_s_b_list = []
signal_eff_ML_list = []
background_eff_ML_list = []

for threshold in thresholds:
    
    signal_counts_after = pred_df.loc[(pred_df.y_pred_proba >= threshold) & (pred_df.y_true == 1), 'y_true'].count()
    signal_counts_before = pred_df.loc[pred_df.y_true == 1, 'y_true'].count()
    background_counts_after = pred_df.loc[(pred_df.y_pred_proba >= threshold) & (pred_df.y_true == 0), 'y_true'].count()
    background_counts_before = pred_df.loc[pred_df.y_true == 0, 'y_true'].count()

    signal_eff_ML = signal_counts_after / signal_counts_before
    background_eff_ML = background_counts_after / background_counts_before

    total_sig_efficiency = sig_eff_presel * signal_eff_ML
    total_bg_suppression = bg_eff_presel * background_eff_ML

    sig_mass_distr = raw_df_list[1].loc[raw_df['tag'] == 'Sig', 'mass_Lc']
    bg_mass_distr = raw_df_list[1].loc[raw_df['tag'] == 'Bg', 'mass_Lc']

    overall_s_b, overall_s_sqrt_s_b, *_= signal_estimates(
        sig_mass_distr=sig_mass_distr,
        bg_mass_distr=bg_mass_distr,
        have_sig_events=CFG.have_sig_events_2 / total_sig_efficiency, 
        have_bg_events=CFG.have_bg_events_2 / total_bg_suppression,
        mass_interval=CFG.mass_interval, 
        visualization=False,
        verbose=False
    )
    
    f1_scores.append(f1_score(y_test, np.where(y_pred_proba_test > threshold, 1, 0)))
    precision_scores.append(precision_score(y_test, np.where(y_pred_proba_test > threshold, 1, 0)))
    s_over_b.append(overall_s_b)
    overall_s_sqrt_s_b_list.append(overall_s_sqrt_s_b)
    signal_eff_ML_list.append(signal_eff_ML)
    background_eff_ML_list.append(background_eff_ML)
    
res_df = pd.DataFrame({
    'threshold': thresholds,
    'f1_score': f1_scores,
    'precision': precision_scores,
    's_b_ratio': s_over_b,
    'significance': overall_s_sqrt_s_b_list,
    'signal_eff_ML': signal_eff_ML_list,
    'background_eff_ML': background_eff_ML_list,
})

res_df

In [ ]:
optimal_threshold = 0.9

#### Physics Evaluation

In [ ]:
from analysis_scripts.estimate_scripts import signal_estimates

threshold = optimal_threshold

pred_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred_proba': y_pred_proba_test
})

signal_counts_after = pred_df.loc[(pred_df.y_pred_proba >= threshold) & (pred_df.y_true == 1), 'y_true'].count()
signal_counts_before = pred_df.loc[pred_df.y_true == 1, 'y_true'].count()
background_counts_after = pred_df.loc[(pred_df.y_pred_proba >= threshold) & (pred_df.y_true == 0), 'y_true'].count()
background_counts_before = pred_df.loc[pred_df.y_true == 0, 'y_true'].count()

signal_eff_ML = signal_counts_after / signal_counts_before
background_eff_ML = background_counts_after / background_counts_before

total_sig_efficiency = sig_eff_presel * signal_eff_ML
total_bg_suppression = bg_eff_presel * background_eff_ML

try:
    experiment.log_metrics({
        'signal_eff_ML': signal_eff_ML,
        'background_eff_ML': background_eff_ML
    })
except NameError:
    pass

print(f'ML signal efficiency: {signal_eff_ML}')
print(f'ML background efficiency: {background_eff_ML}')
print(f'Total signal efficiency: {total_sig_efficiency}')
print(f'Total background efficiency: {total_bg_suppression}')

sig_mass_distr = raw_df_list[1].loc[raw_df_list[1]['tag'] == 'Sig', 'mass_Lc']
bg_mass_distr = raw_df_list[1].loc[raw_df_list[1]['tag'] == 'Bg', 'mass_Lc']

_ = signal_estimates(
    sig_mass_distr=sig_mass_distr,
    bg_mass_distr=bg_mass_distr,
    have_sig_events=CFG.have_sig_events_2 / total_sig_efficiency, 
    have_bg_events=CFG.have_bg_events_2 / total_bg_suppression,
    mass_interval=CFG.mass_interval, 
    visualization=False,
    verbose=True
)


#### Background shape check

In [ ]:
from scipy import stats


threshold = optimal_threshold
alpha = 0.05

df = proc_df_list[1][['mass_Lc']].copy()
df['y_true'] = y_test
df['y_pred_proba'] = y_pred_proba_test

bkg_before_ML = df.loc[df.y_true == 0, 'mass_Lc']
bkg_after_Ml = df.loc[(df.y_pred_proba >= threshold) & (df.y_true == 0), 'mass_Lc']

# kolmogorov-Smirnov test
_, p_value = stats.ks_2samp(bkg_before_ML, bkg_after_Ml, alternative='two-sided')

try:
    experiment.log_metric(name='KS-test p-value', value=p_value)
except NameError:
    pass

if p_value >= alpha:
    print(f'KS-test: distributions are identical, ({alpha=}, {p_value=:0.3f})')
else:
    print(f'KS-test: distributions are different, ({alpha=}, {p_value=:0.3f})')


fig, ax = plt.subplots(figsize=(6, 6))

sns.histplot(
    data=bkg_before_ML,
    label='Bkg before ML',
    element='step',
    fill=False,
    stat='density',
    # alpha=0.8,
    bins=80,
    ax=ax
)

sns.histplot(
    data=bkg_after_Ml,
    label='Bkg after ML',
    element='step',
    fill=False,
    stat='density',
    # alpha=0.8,
    bins=80,
    ax=ax
)

plt.legend(title="Data type:", loc= "best")

ax.set_title('Background distribution', fontsize=10, fontweight='bold')
ax.set_xlabel('Mass Lc [GeV]', fontweight='bold')
ax.set_ylabel('Normalized', fontweight='bold')
ax.grid(True, alpha=0.8, linestyle='--')

try:
    experiment.log_figure(
        figure_name=f"Bkg distr ML",
        figure=plt.gcf()
    )
except NameError:
    pass

plt.show()


# Cut-based ending

### Get ML-handled data

In [ ]:
y_pred_test = np.where(y_pred_proba_test >= optimal_threshold, 1, 0) 

df = proc_df_list[1].copy()
df['pred_tag'] = y_pred_test.copy()

df = df[df['pred_tag'] == 1]

### Create optimal selection path

In [ ]:
from analysis_scripts.selection_scripts import create_best_selection_path


search_columns = [
    'cosAngle_momentum_Lc_sum_momentum_xy',
    'chi2_Lc_PV_xy', 'dist_Lc_PV_xy', 'dist_Lc_PV_xy_custom',
    'chi2_p_PV_xy', 'dist_p_PV_xy', 'dist_p_PV_xy_custom', 'chi2_K_PV_xy',
    'dist_K_PV_xy', 'dist_K_PV_xy_custom', 'chi2_pip_PV_xy',
    'dist_pip_PV_xy', 'dist_pip_PV_xy_custom', 'chi2_p_Lc_xy',
    'dist_p_Lc_xy', 'dist_p_Lc_xy_custom', 'chi2_K_Lc_xy', 'dist_K_Lc_xy',
    'dist_K_Lc_xy_custom', 'chi2_pip_Lc_xy', 'dist_pip_Lc_xy',
    'dist_pip_Lc_xy_custom', 'chi2_Lc', 'chi2_K_pip_xy', 'dist_K_pip_xy',
    'dist_K_pip_xy_custom', 'chi2_p_K_xy', 'dist_p_K_xy',
    'dist_p_K_xy_custom', 'chi2_p_pip_xy', 'dist_p_pip_xy',
    'dist_p_pip_xy_custom',
    'P_Lc', 'Pt_Lc', 'OA_p', 'OA_K', 'OA_pip',
    'l_over_dl_XY', 'ctau_Lc'
]

result_df = create_best_selection_path(
    df=df,
    features=search_columns,
    n_features_to_use=None,
    metric_type='tpr_fpr',
    have_sig_events=CFG.have_sig_events_2,
    have_bg_events=CFG.have_bg_events_2, 
    mass_interval=CFG.mass_interval,
    direction_restrictions={
        'lengthXY_Lc': 'right',
        'dlengthXY_Lc': 'right',
        'l_over_dl_XY': 'right',
        'ctau_Lc': 'right',
        'P_Lc': 'right',
        'Pt_Lc': 'right',
        'cosAngle_momentum_Lc_sum_momentum_xy': 'right',
        'chi2_Lc_PV_xy': 'left',
        'dist_Lc_PV_xy': 'left',
        'dist_Lc_PV_xy_custom': 'left',
        'chi2_p_Lc_xy': 'left',
        'dist_p_Lc_xy': 'left',
        'dist_p_Lc_xy_custom': 'left', 
        'chi2_K_Lc_xy': 'left',
        'dist_K_Lc_xy': 'left',
        'dist_K_Lc_xy_custom': 'left',
        'chi2_pip_Lc_xy': 'left',
        'dist_pip_Lc_xy': 'left',
        'dist_pip_Lc_xy_custom': 'left',
        'chi2_K_pip_xy': 'left',
        'dist_K_pip_xy': 'left',
        'dist_K_pip_xy_custom': 'left',
        'chi2_p_K_xy': 'left',
        'dist_p_K_xy': 'left',
        'dist_p_K_xy_custom': 'left',
        'chi2_p_pip_xy': 'left',
        'dist_p_pip_xy': 'left',
        'dist_p_pip_xy_custom': 'left'
    }
)

In [ ]:
result_df

In [ ]:
cut_based_sig_efficiency = 1
cut_based_bg_suppression = 1

df_copy = df.copy()

cut_based_selection_mask = np.ones(df.shape[0], dtype=bool)